# Compare DD Models With KL / Wasserstein Geometry Metrics

This notebook compares data-driven models against the task-trained ground truth using the new geometry metrics in `Comparison.compute_metrics`.

- `wasserstein_geometry`: lower is better
- `kl_geometry`: lower is better
- `input_source="latents"` compares inferred latents to the reference TT latents after PCA, delay embedding, and linear alignment into a common space


In [ ]:
from ctd.comparison.analysis.tt.tt import Analysis_TT
from ctd.comparison.analysis.dd.dd import Analysis_DD
from ctd.comparison.comparison import Comparison
import dotenv
import os
import numpy as np
import matplotlib.pyplot as plt

dotenv.load_dotenv(override=True)
HOME_DIR = os.getenv("HOME_DIR")
print(HOME_DIR)


In [ ]:
tt_path = HOME_DIR + "content/trained_models/task-trained/tt_3bff/"
tt_analysis = Analysis_TT(run_name="tt_3bff", filepath=tt_path)

tt_analysis.plot_trial_io(num_trials=1)


In [ ]:
# Replace these with your own sweep path / model type if needed.
MODEL_SWEEP_PATH = tt_path + "20250130_NBFF_NODE/"
MODEL_TYPE = "SAE"
MODEL_GROUP = "NODE"

subfolders = sorted([f.path for f in os.scandir(MODEL_SWEEP_PATH) if f.is_dir()])
print(f"Found {len(subfolders)} model folders")
subfolders[:3]


In [ ]:
comparison = Comparison(comparison_tag="3BFF_distribution_metrics")
comparison.load_analysis(tt_analysis, group="TT", reference_analysis=True)

lat_sizes = []
for subfolder in subfolders:
    subfolder = subfolder + "/"
    lat_size = int(subfolder.split("latent_size=")[1].split("_")[0])
    analysis = Analysis_DD.create(
        run_name=f"{MODEL_GROUP}_{lat_size}",
        filepath=subfolder,
        model_type=MODEL_TYPE,
    )
    lat_sizes.append(lat_size)
    comparison.load_analysis(analysis, group=MODEL_GROUP)
    print(f"Loaded {MODEL_GROUP}_{lat_size}")


In [ ]:
geometry_cfg = {
    "input_source": "latents",
    "pca_dim": 16,
    "n_delays": 2,
    "delay_lag": 1,
    "covariance_reg": 1e-5,
    "random_state": 0,
}

metric_dict_list = {
    "state_r2": {},
    "rate_r2": {},
    "wasserstein_geometry": dict(geometry_cfg),
    "kl_geometry": dict(geometry_cfg),
}

metrics = comparison.compute_metrics(metric_dict_list=metric_dict_list)


In [ ]:
lat_sizes = np.array(lat_sizes)
sort_idx = np.argsort(lat_sizes)
lat_sizes = lat_sizes[sort_idx]

metric_names = ["state_r2", "rate_r2", "wasserstein_geometry", "kl_geometry"]
for name in metric_names:
    metrics[name] = np.array(metrics[name])[sort_idx]

mean_metrics = {}
for name in metric_names:
    mean_metrics[name] = np.array([
        [lat, metrics[name][lat_sizes == lat].mean()]
        for lat in np.unique(lat_sizes)
    ])

print("Best latent size by Wasserstein:", lat_sizes[np.argmin(metrics["wasserstein_geometry"])])
print("Best latent size by KL:", lat_sizes[np.argmin(metrics["kl_geometry"])])


In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(14, 10))

ax[0, 0].scatter(lat_sizes, metrics["state_r2"], c=np.log10(lat_sizes))
ax[0, 0].plot(mean_metrics["state_r2"][:, 0], mean_metrics["state_r2"][:, 1], c="k", alpha=0.3)
ax[0, 0].set_xlabel("Latent Size")
ax[0, 0].set_ylabel("State R2")
ax[0, 0].set_title("Ground-Truth State Fit")

ax[0, 1].scatter(lat_sizes, metrics["rate_r2"], c=np.log10(lat_sizes))
ax[0, 1].plot(mean_metrics["rate_r2"][:, 0], mean_metrics["rate_r2"][:, 1], c="k", alpha=0.3)
ax[0, 1].set_xlabel("Latent Size")
ax[0, 1].set_ylabel("Rate R2")
ax[0, 1].set_title("Ground-Truth Rate Fit")

ax[1, 0].scatter(lat_sizes, metrics["wasserstein_geometry"], c=np.log10(lat_sizes))
ax[1, 0].plot(
    mean_metrics["wasserstein_geometry"][:, 0],
    mean_metrics["wasserstein_geometry"][:, 1],
    c="k",
    alpha=0.3,
)
ax[1, 0].set_xlabel("Latent Size")
ax[1, 0].set_ylabel("Wasserstein Geometry")
ax[1, 0].set_title("Lower Is Better")

scatter = ax[1, 1].scatter(lat_sizes, metrics["kl_geometry"], c=np.log10(lat_sizes))
ax[1, 1].plot(mean_metrics["kl_geometry"][:, 0], mean_metrics["kl_geometry"][:, 1], c="k", alpha=0.3)
ax[1, 1].set_xlabel("Latent Size")
ax[1, 1].set_ylabel("KL Geometry")
ax[1, 1].set_title("Lower Is Better")

cbar = plt.colorbar(scatter, ax=ax.ravel().tolist())
cbar.set_label("Latent Size")
cbar.set_ticks(np.log10(np.unique(lat_sizes)))
cbar.set_ticklabels(np.unique(lat_sizes))

plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

ax[0].scatter(metrics["state_r2"], metrics["wasserstein_geometry"], c=np.log10(lat_sizes))
ax[0].set_xlabel("State R2")
ax[0].set_ylabel("Wasserstein Geometry")
ax[0].set_title("Ground-Truth Fit vs Geometry")

ax[1].scatter(metrics["state_r2"], metrics["kl_geometry"], c=np.log10(lat_sizes))
ax[1].set_xlabel("State R2")
ax[1].set_ylabel("KL Geometry")
ax[1].set_title("Ground-Truth Fit vs Geometry")

plt.tight_layout()
plt.show()
